# V12 Engines — Delivery-Delay Preprocessing & Feature Template
**Owner:** Data & preprocessing · **Status:** template for the event (follow the team docs)

This notebook turns the raw USAID SCMS CSV into a model-ready feature table that matches the **Interface & Contract Spec**. Every decision here is from the **Project Guidelines v2** data audit.

> **Rules note:** this is a *template / plan* for understanding and practice. The final production run happens on LinuxONE during the event. Keep it honest: no fabricated numbers, leakage-safe by construction.

**Confirmed decisions baked in:** label = delivered > scheduled · keep From RDC · shipment-mode blanks → "Unknown" · decision date = PO else PQ (fallback scheduled for cutoff) · split by scheduled date Option B (train ≤2013 / test 2014–2015) · weight excluded · prior-late-rate = leakage-safe, 365-day window, smoothing M=20, lane = each of country/vendor/route.

## 0. Config

In [3]:
import pandas as pd, numpy as np

CSV_PATH       = "SCMS_Delivery_History_Dataset_20150929.csv"  # <-- set your path
WINDOW_DAYS    = 365      # recency window for prior late-rate
SMOOTH_M       = 20.0     # smoothing strength (pseudo-counts)
FALLBACK_PRIOR = 0.115    # overall late rate; used only when no prior outcomes exist
TEST_YEARS     = {2014, 2015}   # Option B: train <=2013, test 2014-2015

## 1. Load & parse dates
Two date formats in this file: scheduled/delivered are `%d-%b-%y` (e.g. 10-Jan-15); PO/PQ are `%m/%d/%Y` with placeholder text ("N/A - From RDC", "Pre-PQ Process", "Date Not Captured") that becomes NaT.

In [4]:
df = pd.read_csv(CSV_PATH, encoding="latin-1", low_memory=False)
df.columns = [c.strip().lower() for c in df.columns]

scheduled = pd.to_datetime(df["scheduled delivery date"], errors="coerce", format="%d-%b-%y")
delivered = pd.to_datetime(df["delivered to client date"], errors="coerce", format="%d-%b-%y")
po = pd.to_datetime(df["po sent to vendor date"],  errors="coerce", format="%m/%d/%Y")
pq = pd.to_datetime(df["pq first sent to client date"], errors="coerce", format="%m/%d/%Y")

print("rows:", len(df), "| scheduled NaT:", scheduled.isna().sum(), "| delivered NaT:", delivered.isna().sum())

rows: 10324 | scheduled NaT: 0 | delivered NaT: 0


## 2. Label
`late = delivered after scheduled`. `days_late` is the regressor target (clipped at 0). The order/decision date is **not** part of the label — it is only used for the split and the runway.

In [5]:
days_late  = (delivered - scheduled).dt.days
label_late = (days_late > 0).astype(int)
days_late  = days_late.clip(lower=0)
print("late rate: %.1f%% (%d of %d)" % (label_late.mean()*100, label_late.sum(), len(df)))

late rate: 11.5% (1186 of 10324)


## 3. Decision-date anchor
`decision_date = PO sent to vendor, else PQ first sent` (covers ~81%). For the leakage-safe cutoff we fall back to `scheduled` when neither exists, and flag it with `decision_date_known`.

In [6]:
decision_date       = po.fillna(pq)
decision_date_known = decision_date.notna()
cutoff              = decision_date.fillna(scheduled)   # time-safe scoring cutoff
print("decision_date known: %.1f%%" % (decision_date_known.mean()*100))

decision_date known: 81.3%


## 4. Categorical & numeric fields
Shipment-mode blanks become **"Unknown"** (NOT imputed to Ocean — the data audit showed blanks are 1.1% late vs Ocean 17.5%, so imputing would mislabel).

In [7]:
out = pd.DataFrame()
out["order_id"]           = (df["id"] if "id" in df.columns else pd.Series(range(len(df)))).astype(str)
out["country"]            = df["country"].astype(str)
out["vendor"]             = df["vendor"].astype(str)
out["fulfillment_route"]  = df["fulfill via"].astype(str)         # Direct Drop | From RDC
out["shipment_mode"]      = df["shipment mode"].fillna("Unknown").replace({"": "Unknown"}).astype(str)
out["product_group"]      = df["product group"].astype(str)
out["managed_by"]         = df["managed by"].astype(str)
out["manufacturing_site"] = df["manufacturing site"].astype(str)
out["quantity"]           = pd.to_numeric(df["line item quantity"], errors="coerce")

## 5. Planned lead time
`planned_lead_time_days = scheduled − decision_date` (known for ~81%). A few rows where the anchor is after the scheduled date (data error) → set to NaN. Weight is **excluded** (not reliably known at order time).

In [8]:
plt_days = (scheduled - decision_date).dt.days
plt_days = plt_days.where(plt_days >= 0)   # drop negative (anchor after scheduled)
out["planned_lead_time_days"] = plt_days
out["decision_date_known"]    = decision_date_known.astype(int)

out["decision_date"]           = decision_date
out["scheduled_delivery_date"] = scheduled
out["label_late"]              = label_late
out["days_late"]               = days_late

## 6. Leakage-safe prior late-rate (365-day window + smoothing)
For each shipment, the prior late-rate of a key (country / vendor / route) uses **only shipments that had already been DELIVERED before this shipment's cutoff**, within the last 365 days, smoothed toward the same-window global prior. This is leakage-safe by construction (the current row and any future shipment can never enter). The data is non-stationary (late rates swing by year), so the recency window matters — it roughly doubled class separation vs an all-history average.

In [9]:
def add_prior(key, window_days=WINDOW_DAYS, M=SMOOTH_M, fb=FALLBACK_PRIOR):
    cut = cutoff.values.astype("datetime64[D]")
    dlv = delivered.values.astype("datetime64[D]")
    lab = label_late.values
    kv  = out[key].values
    N   = len(out); res = np.empty(N); known = np.zeros(N, dtype=int)
    w   = np.timedelta64(window_days, "D")
    go = np.argsort(dlv); gd = dlv[go]; gl = lab[go]; gc = np.concatenate([[0], np.cumsum(gl)])
    def win(sd, cum, c):
        hi = np.searchsorted(sd, c, "left"); lo = np.searchsorted(sd, c - w, "left")
        return hi - lo, cum[hi] - cum[lo]
    tmp = pd.DataFrame({"k": kv, "cut": cut, "dlv": dlv, "lab": lab})
    for _, g in tmp.groupby("k"):
        idx = g.index.values
        o = np.argsort(g["dlv"].values); kd = g["dlv"].values[o]; kl = g["lab"].values[o]
        kc = np.concatenate([[0], np.cumsum(kl)])
        for i in idx:
            c = tmp["cut"].values[i]; sd = tmp["dlv"].values[i]; sl = tmp["lab"].values[i]
            n, s = win(kd, kc, c)
            if (sd < c) and (sd >= c - w):       # exclude self if inside the window
                n -= 1; s -= sl
            gn, gs = win(gd, gc, c); gp = gs/gn if gn > 0 else fb
            res[i] = (s + M*gp) / (n + M) if (n + M) > 0 else fb
            known[i] = 1 if n > 0 else 0
    return res, known

for key in ["country", "vendor", "fulfillment_route"]:
    r, kn = add_prior(key)
    out[f"prior_late_rate_{key}"] = r
    out[f"prior_known_{key}"]     = kn

print("prior by route:", out.groupby("fulfillment_route")["prior_late_rate_fulfillment_route"].mean().round(3).to_dict())

prior by route: {'Direct Drop': 0.054, 'From RDC': 0.137}


## 7. Time-based split (Option B)
Split on the always-present `scheduled` date: train ≤2013, test 2014–2015 (≈25% test, ~356 late cases — enough for stable PR-AUC / recall).

In [10]:
sy = scheduled.dt.year
out["_split"] = np.where(sy.isin(TEST_YEARS), "test", "train")
print(out["_split"].value_counts().to_dict())

{'train': 7779, 'test': 2545}


## 8. Assemble feature matrix (matches the Interface Spec) & sanity checks

In [ ]:
FEATURES = ["country","vendor","fulfillment_route","shipment_mode","product_group",
            "managed_by","manufacturing_site","quantity","planned_lead_time_days","decision_date_known",
            "prior_late_rate_country","prior_late_rate_vendor","prior_late_rate_fulfillment_route",
            "prior_known_country","prior_known_vendor","prior_known_fulfillment_route"]

train = out[out._split=="train"]; test = out[out._split=="test"]
X_train, y_train = train[FEATURES], train["label_late"]
X_test,  y_test  = test[FEATURES],  test["label_late"]
print("train", X_train.shape, "late", int(y_train.sum()))
print("test ", X_test.shape,  "late", int(y_test.sum()))
print("NaN only in planned_lead_time?", out[FEATURES].isna().sum()[lambda s: s>0].to_dict())

# Save the cleaned tables (parquet):
train.to_parquet("train.parquet")
test.to_parquet("test.parquet")
print("saved train.parquet / test.parquet")

SyntaxError: invalid syntax (1736836126.py, line 13)

## 9. Honesty / handoff notes
- **No leakage:** prior features use only outcomes delivered before each row's cutoff; verified on the audit data.
- **Non-stationarity is real** (late rates swing by year) → the 365-day window is deliberate, and it is also the pitch argument for *retraining on current data* in deployment.
- **planned_lead_time NaN** for ~19% (no decision date) is expected — the model should treat it as missing with the `decision_date_known` flag, never drop silently.
- **Next (model owner):** class-weighted Logistic Regression baseline + Snap ML boosted-tree classifier (late prob) and regressor (days late); evaluate with PR-AUC and recall at the operating threshold, not accuracy; report the Error breakdown (Type I / Type II).